# Gurgaon Real Estate — Complete, Consolidated EDA
A single, self-contained notebook that reproduces and consolidates the full exploratory data analysis (EDA) you described across your multiple Colab notebooks and report.

**How to use this notebook**
1. Run the first cell to install (optional) extras if needed (skip if offline).
2. Update `CSV_PATH` if your CSV is elsewhere. By default it expects `Gurgaon_RealEstate.csv` in the same folder.
3. Execute cells in order. Each section prints intermediate summaries and makes *one plot per cell* (no subplots), per your guidelines.
4. At the end, a cleaned CSV is saved next to the notebook.

In [ ]:

# (Optional) Lightweight installs – skip safely if offline
# If you are offline, these will simply fail gracefully and the rest will still run.
try:
    import pip
    # Nothing critical to install; placeholder in case you want to add 'missingno' etc.
except Exception as e:
    print("Skipping optional installs:", e)

## 1) Imports & Globals

In [ ]:

import os
from pathlib import Path
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict
from typing import Dict, List, Tuple, Optional

# If SciPy is available, we use it for z-scores & dendrograms
try:
    from scipy import stats
    from scipy.cluster.hierarchy import linkage, dendrogram
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

CSV_PATH = "Gurgaon_RealEstate.csv"  # change if needed

# Matplotlib defaults – keep things simple and consistent
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True
})

def safe_show():
    """Helper to ensure each figure is finalized and released."""
    plt.tight_layout()
    plt.show()
    plt.close()

## 2) Load Data & Basic Overview

In [ ]:

# Try to load CSV from working directory; fall back to /mnt/data if present there.
candidate_paths = [Path(CSV_PATH), Path('/mnt/data')/CSV_PATH]
csv_file = None
for p in candidate_paths:
    if p.exists():
        csv_file = p
        break

if csv_file is None:
    raise FileNotFoundError(f"Could not find {CSV_PATH}. Place it next to the notebook or in /mnt/data.")

df = pd.read_csv(csv_file)
df_original = df.copy(deep=True)

print("Shape (rows, cols):", df.shape)
print("\nInfo:")
print(df.info())

print("\nPreview:")
display(df.head(10))

## 3) Duplicate Handling

In [ ]:

dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("New shape after dropping duplicates:", df.shape)
else:
    print("No duplicate rows found.")

## 4) Column Normalization Helper (auto-map real column names)

In [ ]:

# We normalize column names to make downstream code robust to slight naming differences.
def norm_name(s: str) -> str:
    return (
        str(s).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(".", "")
    )

df.columns = [norm_name(c) for c in df.columns]

# Expected fields with multiple candidate names
CANDIDATES = {
    "property_type": ["property_type", "type", "prop_type"],
    "society": ["society", "project", "apartment", "builder"],
    "sector": ["sector", "locality", "area_name"],
    "price": ["price", "price_lakh", "price_in_lakhs", "total_price"],
    "price_per_sqft": ["price_per_sqft", "price_sqft", "ppsqft", "psf"],
    "area": ["area", "total_area", "plot_area"],
    "bedroom": ["bedroom", "bhk", "beds"],
    "bathroom": ["bathroom", "baths", "bathrooms"],
    "balcony": ["balcony", "balconies"],
    "floor_num": ["floor_num", "floor", "floor_number"],
    "facing": ["facing", "direction", "face"],
    "built_up_area": ["built_up_area", "builtup_area"],
    "super_built_up_area": ["super_built_up_area", "super_builtup_area", "super_area"],
    "carpet_area": ["carpet_area", "carpet"],
    "pooja_room": ["pooja_room", "prayer_room", "puja_room"],
    "study_room": ["study_room", "study"],
    "storeroom": ["storeroom", "store_room", "store"],
    "servant_room": ["servant_room", "maid_room"],
    "furnishing_type": ["furnishing_type", "furnish", "furnishing"],
    "luxury_score": ["luxury_score", "luxuryindex", "luxury_index", "amenity_score"]
}

def find_col(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    cols = list(df.columns)
    nmap = {c: norm_name(c) for c in cols}
    inv = {v:k for k,v in nmap.items()}
    norm_cols = set(nmap.values())
    for cand in names:
        nc = norm_name(cand)
        if nc in norm_cols:
            return inv[nc]
    # fallback: substring heuristic
    for cand in names:
        nc = norm_name(cand)
        for c in norm_cols:
            if nc in c:
                return inv[c]
    return None

COLS: Dict[str, Optional[str]] = {k: find_col(df, v) for k,v in CANDIDATES.items()}
pd.DataFrame([COLS]).T.rename(columns={0:"mapped_to"})

## 5) Basic Descriptives (shape, describe, dtypes)

In [ ]:

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)

print("\nNumeric describe():")
display(df.select_dtypes(include=[np.number]).describe())

print("\nCategorical cardinality:")
cat_cols = [c for c in df.columns if df[c].dtype == 'object' or str(df[c].dtype).startswith("category")]
display(pd.DataFrame({
    "column": cat_cols,
    "nunique": [df[c].nunique(dropna=True) for c in cat_cols]
}))

## 6) Missing Data — Counts & %

In [ ]:

miss_cnt = df.isna().sum()
miss_pct = df.isna().mean()*100
missing_summary = pd.DataFrame({"missing_count": miss_cnt, "missing_percent": miss_pct}).sort_values("missing_percent", ascending=False)
display(missing_summary)

# Simple heatmap of missingness (matplotlib only)
mask = df.isna().astype(int)
plt.imshow(mask, aspect='auto', interpolation='nearest')
plt.title("Missingness Matrix (1 = missing)")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.xticks(range(len(df.columns)), df.columns, rotation=90)
safe_show()

# Dendrogram of missingness patterns (if SciPy available)
if SCIPY_OK and df.shape[1] >= 2:
    # Correlation of missingness by columns
    miss_corr = np.corrcoef(mask.T)
    # Replace NaN correlations resulting from constant columns
    miss_corr = np.nan_to_num(miss_corr, nan=0.0)
    Z = linkage(miss_corr, method='average')
    plt.figure(figsize=(10, 4))
    dendrogram(Z, labels=df.columns, leaf_rotation=90)
    plt.title("Dendrogram of Missingness Pattern (by column)")
    safe_show()
else:
    print("SciPy not available or not enough columns – skipping dendrogram.")

## 7) Complete Case Analysis (CCA) for small-missing columns

In [ ]:

# Identify columns with small missingness (0–5%)
small_miss_cols = [c for c in df.columns if 0 < df[c].isna().mean() <= 0.05]
print("Columns eligible for CCA (0–5% missing):", small_miss_cols)

df_cca = df.dropna(subset=small_miss_cols) if small_miss_cols else df.copy()
print("Before CCA:", df.shape, "After CCA:", df_cca.shape)

# Compare distributions numerically via KS-test (if SciPy available)
if SCIPY_OK:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    rows = []
    for c in numeric_cols:
        base = df[c].dropna().values
        cca  = df_cca[c].dropna().values
        if len(base) > 10 and len(cca) > 10:
            stat, p = stats.ks_2samp(base, cca)
            rows.append((c, stat, p))
    if rows:
        ks_df = pd.DataFrame(rows, columns=["column","ks_statistic","p_value"])
        display(ks_df.sort_values("p_value", ascending=False))
else:
    print("SciPy not available – skipping KS-test summary.")

## 8) Helper Plotters (matplotlib only, one plot per cell)

In [ ]:

def plot_bar(series: pd.Series, title: str, xlabel: str, ylabel: str):
    series = series.sort_values(ascending=False)
    plt.figure()
    series.plot(kind='bar')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    safe_show()

def plot_pie(series: pd.Series, title: str):
    plt.figure()
    series.plot(kind='pie', autopct='%1.1f%%')
    plt.title(title)
    safe_show()

def plot_hist(arr: pd.Series, bins=30, title="Histogram", xlabel="", ylabel="Count"):
    arr = arr.dropna()
    plt.figure()
    plt.hist(arr, bins=bins, alpha=0.8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    safe_show()

def plot_box(arr: pd.Series, title="Box Plot", ylabel=""):
    arr = arr.dropna()
    plt.figure()
    plt.boxplot(arr.values, vert=True, labels=[ylabel if ylabel else ""])
    plt.title(title)
    safe_show()

def plot_scatter(x: pd.Series, y: pd.Series, title: str, xlabel: str, ylabel: str, jitter_x=False):
    # Optional jitter for categorical-like x
    x_vals = x.copy()
    if jitter_x:
        # add small random jitter
        x_vals = x_vals.astype(float) + (np.random.rand(len(x_vals)) - 0.5)*0.15
    plt.figure()
    plt.scatter(x_vals, y, alpha=0.6)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    safe_show()

## 9) Property Type — Distribution (bar & pie)

In [ ]:

pt_col = COLS.get("property_type")
if pt_col and pt_col in df.columns:
    vc = df[pt_col].astype('category').value_counts(dropna=False)
    display(vc.to_frame("count"))
    plot_bar(vc, "Property Type Distribution", "Type", "Count")
    plot_pie(vc, "Property Type Distribution (%)")
else:
    print("property_type column not found – skipping this section.")

## 10) Society — Frequency, Rare Handling, Top-15

In [ ]:

soc_col = COLS.get("society")
if soc_col and soc_col in df.columns:
    vc = df[soc_col].astype('category').value_counts()
    display(vc.to_frame("count").head(30))

    # Filter rare (<=15) to 'other_flats'
    threshold = 15
    rare_mask = df[soc_col].map(vc).fillna(0) <= threshold
    df['society_filtered'] = df[soc_col].where(~rare_mask, other="Other Flats")

    # Plot societies > threshold
    filtered_counts = df.loc[df['society_filtered']!="Other Flats", 'society_filtered'].value_counts()
    plot_bar(filtered_counts, f"Societies (> {threshold})", "Society", "Count")

    # Top 15 excluding independent & Other Flats (case-insensitive contains)
    exclude = {"independent", "other flats"}
    top15 = (df.loc[~df['society_filtered'].str.lower().isin(exclude), 'society_filtered']
               .value_counts().head(15))
    plot_bar(top15, "Top-15 Societies (Excl. 'Independent' & 'Other Flats')", "Society", "Count")
else:
    print("society column not found – skipping this section.")

## 11) Sector — Top-15 & Frequency Bins

In [ ]:

sec_col = COLS.get("sector")
if sec_col and sec_col in df.columns:
    vc = df[sec_col].astype('category').value_counts()
    top15 = vc.head(15)
    plot_bar(top15, "Top-15 Sectors", "Sector", "Count")

    # Frequency-bin bar: how many sectors fall into each frequency range
    counts = vc.values
    bins = [0, 5, 10, 20, 50, 100, counts.max()]
    labels = ["1–5","6–10","11–20","21–50","51–100", f"100+"]
    freq_bins = pd.cut(counts, bins=bins, labels=labels, include_lowest=True, right=True)
    freq_summary = pd.Series(freq_bins).value_counts().sort_index()
    plot_bar(freq_summary, "Sector Frequency Bins", "Bin", "Num Sectors")
else:
    print("sector column not found – skipping this section.")

## 12) Price — Summary, Histogram, Box, Bins, Skew/Kurtosis

In [ ]:

price_col = COLS.get("price")
if price_col and price_col in df.columns:
    print(df[price_col].describe())
    plot_hist(df[price_col], bins=40, title="Price Distribution", xlabel="Price")
    plot_box(df[price_col], title="Price — Box Plot", ylabel="Price")
    if SCIPY_OK:
        print("Skewness:", stats.skew(df[price_col].dropna()))
        print("Kurtosis:", stats.kurtosis(df[price_col].dropna()))

    # Custom bins (edit if needed)
    quantiles = df[price_col].quantile([0, 0.25, 0.5, 0.75, 0.9, 1.0]).values
    # Coarse bins based on quantiles
    bins = np.unique(quantiles)
    labels = [f"Bin {i+1}" for i in range(len(bins)-1)]
    binned = pd.cut(df[price_col], bins=bins, labels=labels, include_lowest=True, duplicates='drop')
    price_bin_counts = binned.value_counts().sort_index()
    plot_bar(price_bin_counts, "Price — Quantile Bins", "Bin", "Count")
else:
    print("price column not found – skipping this section.")

## 13) Price per Sqft — Summary, Histogram, Box, Bins

In [ ]:

ppsf_col = COLS.get("price_per_sqft")
if ppsf_col and ppsf_col in df.columns:
    print(df[ppsf_col].describe())
    plot_hist(df[ppsf_col], bins=60, title="Price per Sqft Distribution", xlabel="Price/Sqft")
    plot_box(df[ppsf_col], title="Price per Sqft — Box Plot", ylabel="Price/Sqft")

    # Bins
    q = df[ppsf_col].quantile([0, 0.25, 0.5, 0.75, 0.9, 1.0]).values
    bins = np.unique(q)
    labels = [f"Bin {i+1}" for i in range(len(bins)-1)]
    binned = pd.cut(df[ppsf_col], bins=bins, labels=labels, include_lowest=True, duplicates='drop')
    plot_bar(binned.value_counts().sort_index(), "PPSF — Quantile Bins", "Bin", "Count")
else:
    print("price_per_sqft column not found – skipping this section.")

## 14) Bedroom — Distribution, Jitter, Crosstab vs Property Type

In [ ]:

bed_col = COLS.get("bedroom")
pt_col = COLS.get("property_type")
if bed_col and bed_col in df.columns:
    print("Mode:", df[bed_col].mode(dropna=True).iloc[0] if not df[bed_col].mode(dropna=True).empty else None)
    print("Missing:", df[bed_col].isna().sum())
    vc = df[bed_col].value_counts().sort_index()
    plot_bar(vc, "Bedrooms — Distribution", "Bedrooms", "Count")
    # Jitter plot (x = bedrooms, y = count proxy)
    # For a strip-like viz, we scatter against zeros with jitter
    x = df[bed_col].dropna()
    y = np.zeros_like(x, dtype=float) + 1.0
    plot_scatter(x, y, "Bedrooms — Strip (jitter on X)", "Bedrooms", "Dummy=1", jitter_x=True)

    if pt_col and pt_col in df.columns:
        ct = pd.crosstab(df[bed_col], df[pt_col])
        display(ct)
        # Stacked style: one bar per bedroom with stacked property types
        # (Plot each property_type sequentially with bottom to emulate stacked bar)
        base = np.zeros(len(ct))
        for col in ct.columns:
            plt.figure()
            plt.bar(ct.index.astype(str), ct[col].values, bottom=base)
            plt.title(f"Bedrooms vs Property Type — adding '{col}' layer")
            plt.xlabel("Bedrooms"); plt.ylabel("Count")
            safe_show()
            base = base + ct[col].values
else:
    print("bedroom column not found – skipping this section.")

## 15) Bathroom — Distribution, Flats-only Hist, Crosstab

In [ ]:

bath_col = COLS.get("bathroom")
pt_col = COLS.get("property_type")
if bath_col and bath_col in df.columns:
    print("Dtype:", df[bath_col].dtype, "| Missing:", df[bath_col].isna().sum())
    vc = df[bath_col].value_counts().sort_index()
    plot_bar(vc, "Bathrooms — Distribution", "Bathrooms", "Count")

    # Strip (jitter x)
    x = df[bath_col].dropna().astype(float)
    y = np.zeros_like(x, dtype=float) + 1.0
    plot_scatter(x, y, "Bathrooms — Strip (jitter on X)", "Bathrooms", "Dummy=1", jitter_x=True)

    if pt_col and pt_col in df.columns:
        flats = df[df[pt_col].str.lower().str.contains("flat", na=False)]
        if not flats.empty:
            plot_hist(flats[bath_col], bins=range(int(flats[bath_col].min()), int(flats[bath_col].max())+2),
                      title="Bathrooms in Flats — Histogram", xlabel="Bathrooms")
        ct = pd.crosstab(df[bath_col], df[pt_col])
        display(ct)
else:
    print("bathroom column not found – skipping this section.")

## 16) Balcony — Summary, Flats-only Hist, Strip, Summary by Type

In [ ]:

bal_col = COLS.get("balcony")
pt_col = COLS.get("property_type")
if bal_col and bal_col in df.columns:
    print(df[bal_col].describe(include='all'))
    print("Missing:", df[bal_col].isna().sum())

    if pt_col and pt_col in df.columns:
        flats = df[df[pt_col].str.lower().str.contains("flat", na=False)]
        if not flats.empty:
            plot_hist(pd.to_numeric(flats[bal_col], errors='coerce'), bins=20,
                      title="Balconies in Flats — Histogram", xlabel="Balcony Count")

    # Generic bar of categories
    vc = df[bal_col].astype(str).value_counts()
    plot_bar(vc, "Balcony — Category Frequencies", "Balcony", "Count")

    # Strip-like
    x = pd.to_numeric(df[bal_col], errors='coerce').dropna()
    y = np.ones_like(x, dtype=float)
    if len(x) > 0:
        plot_scatter(x, y, "Balcony — Strip (jitter on X)", "Balconies", "Dummy=1", jitter_x=True)

    if pt_col and pt_col in df.columns:
        ct = pd.crosstab(df[bal_col], df[pt_col])
        display(ct)
else:
    print("balcony column not found – skipping this section.")

## 17) Floor Number — Distribution, Outliers, vs Property Type

In [ ]:

floor_col = COLS.get("floor_num")
pt_col = COLS.get("property_type")
if floor_col and floor_col in df.columns:
    vc = pd.to_numeric(df[floor_col], errors='coerce').dropna().astype(int).value_counts().sort_index()
    plot_bar(vc, "Floor Number — Distribution", "Floor", "Count")

    x = pd.to_numeric(df[floor_col], errors='coerce').dropna()
    y = np.ones_like(x, dtype=float)
    plot_scatter(x, y, "Floor Number — Strip (jitter on X)", "Floor", "Dummy=1", jitter_x=True)

    if pt_col and pt_col in df.columns:
        ct = pd.crosstab(pd.to_numeric(df[floor_col], errors='coerce'), df[pt_col])
        display(ct)
else:
    print("floor number column not found – skipping this section.")

## 18) Facing — Distribution, % Pie, Type Comparison

In [ ]:

face_col = COLS.get("facing")
pt_col = COLS.get("property_type")
if face_col and face_col in df.columns:
    vc = df[face_col].astype('category').value_counts()
    plot_bar(vc, "Facing — Distribution", "Facing", "Count")
    plot_pie(vc, "Facing — Percentage")

    # Strip-like per category: map categories to ints
    cat_map = {cat:i for i,cat in enumerate(vc.index)}
    x = df[face_col].map(cat_map).dropna()
    y = np.ones_like(x, dtype=float)
    if len(x) > 0:
        plot_scatter(x, y, "Facing — Strip (category index)", "Category Index", "Dummy=1", jitter_x=True)

    if pt_col and pt_col in df.columns:
        ct = pd.crosstab(df[face_col], df[pt_col])
        display(ct)
else:
    print("facing column not found – skipping this section.")

## 19) Built-up / Super Built-up / Carpet Area — Summary, Bins, Type Comparison

In [ ]:

bua = COLS.get("built_up_area")
sba = COLS.get("super_built_up_area")
car = COLS.get("carpet_area")
pt_col = COLS.get("property_type")

def impute_areas(df: pd.DataFrame, bua, sba, car):
    # Conservative ratio-based imputation using medians from available rows
    df = df.copy()
    if not (bua and sba and car):
        return df

    # Compute ratios where both present
    r_bua_sba = df[[bua, sba]].dropna()
    r_car_sba = df[[car, sba]].dropna()

    med_bua_over_sba = (r_bua_sba[bua] / r_bua_sba[sba]).median() if not r_bua_sba.empty else None
    med_car_over_sba = (r_car_sba[car] / r_car_sba[sba]).median() if not r_car_sba.empty else None

    # Case 1: sba present, others missing
    if sba in df.columns:
        if bua and bua in df.columns and med_bua_over_sba is not None:
            idx = df[bua].isna() & df[sba].notna()
            df.loc[idx, bua] = df.loc[idx, sba] * med_bua_over_sba
        if car and car in df.columns and med_car_over_sba is not None:
            idx = df[car].isna() & df[sba].notna()
            df.loc[idx, car] = df.loc[idx, sba] * med_car_over_sba

    # Case 2: bua present, sba missing → invert ratio
    if bua and bua in df.columns and sba and sba in df.columns and med_bua_over_sba and med_bua_over_sba != 0:
        idx = df[sba].isna() & df[bua].notna()
        df.loc[idx, sba] = df.loc[idx, bua] / med_bua_over_sba

    return df

# Describe & missing
for c in [bua, sba, car]:
    if c and c in df.columns:
        print(f"\n[{c}] describe:")
        display(df[c].describe())

# Impute area trio in a working copy
df_area = impute_areas(df, bua, sba, car)

# Bin built_up_area if present
if bua and bua in df_area.columns:
    q = df_area[bua].dropna().quantile([0, 0.25, 0.5, 0.75, 0.9, 1.0]).values
    bins = np.unique(q)
    labels = [f"Very Low", "Low", "Medium", "High", "Very High"][: len(bins)-1]
    binned = pd.cut(df_area[bua], bins=bins, labels=labels, include_lowest=True, duplicates='drop')
    plot_bar(binned.value_counts().sort_index(), "Built-up Area — Quantile Bins", "Bin", "Count")

    # Box
    plot_box(df_area[bua], title="Built-up Area — Box Plot", ylabel="Built-up Area")

# Type comparison
if pt_col and bua and pt_col in df_area.columns and bua in df_area.columns:
    agg = df_area.groupby(pt_col)[bua].describe()
    display(agg)

## 20) Binary Features — Pooja/Study/Storeroom/Servant

In [ ]:

bin_cols = [COLS.get("pooja_room"), COLS.get("study_room"), COLS.get("storeroom"), COLS.get("servant_room")]
bin_cols = [c for c in bin_cols if c and c in df.columns]

for c in bin_cols:
    # Coerce to {0,1}
    vals = pd.to_numeric(df[c], errors='coerce')
    # If values are not {0,1}, map truthy to 1
    if not set(np.unique(vals.dropna())).issubset({0,1}):
        vals = (vals > 0).astype(int)
    vc = vals.value_counts().sort_index()
    print(f"\n[{c}] value counts:")
    display(vc.to_frame("count"))
    plot_pie(vc, f"{c} — Presence (%)")

## 21) Furnishing Type — Distribution, Per Type Summary

In [ ]:

fur_col = COLS.get("furnishing_type")
pt_col = COLS.get("property_type")
if fur_col and fur_col in df.columns:
    vc = df[fur_col].astype(str).value_counts()
    plot_bar(vc, "Furnishing Type — Distribution", "Furnishing", "Count")
    if pt_col and pt_col in df.columns:
        display(pd.crosstab(df[fur_col], df[pt_col]))
else:
    print("furnishing_type column not found – skipping this section.")

## 22) Multivariate — Price vs (BuiltUp, Bathrooms, Bedrooms, Type, Furnishing, Balcony, Floor, Luxury)

In [ ]:

price = COLS.get("price")
bua = COLS.get("built_up_area")
bath = COLS.get("bathroom")
bed = COLS.get("bedroom")
pt  = COLS.get("property_type")
fur = COLS.get("furnishing_type")
bal = COLS.get("balcony")
flr = COLS.get("floor_num")
lux = COLS.get("luxury_score")

def safe_series(col):
    return pd.to_numeric(df[col], errors='coerce') if col and col in df.columns else None

if price and price in df.columns:
    s_price = safe_series(price)

    for (col, label) in [(bua,"Built-up Area"),
                         (bath,"Bathrooms"),
                         (bed,"Bedrooms"),
                         (bal,"Balconies"),
                         (flr,"Floor Num"),
                         (lux,"Luxury Score")]:
        s_col = safe_series(col)
        if s_col is not None:
            plot_scatter(s_col, s_price, f"Price vs {label}", label, "Price")

    # Price vs Property Type (box-plot-like via scatter with jitter)
    if pt and pt in df.columns:
        cat_map = {cat:i for i,cat in enumerate(df[pt].astype('category').cat.categories)}
        x = df[pt].map(cat_map)
        plot_scatter(x, s_price, "Price vs Property Type (category index)", "Type Index", "Price")

    # Price vs Furnishing
    if fur and fur in df.columns:
        cat_map = {cat:i for i,cat in enumerate(df[fur].astype('category').cat.categories)}
        x = df[fur].map(cat_map)
        plot_scatter(x, s_price, "Price vs Furnishing (category index)", "Furnishing Index", "Price")
else:
    print("price column not found – skipping multivariate plots.")

## 23) Outliers — Detection (IQR & Z-score) and Handling

In [ ]:

def iqr_outlier_mask(s: pd.Series, k: float = 1.5):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if s.empty:
        return pd.Series([], dtype=bool)
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return (s < lo) | (s > hi)

def zscore_outlier_mask(s: pd.Series, z: float = 3.0):
    s = pd.to_numeric(s, errors='coerce')
    if s.dropna().empty or not SCIPY_OK:
        return pd.Series([False]*len(s))
    zs = np.abs(stats.zscore(s.dropna()))
    mask = pd.Series(False, index=s.dropna().index)
    mask.iloc[np.where(zs > z)[0]] = True
    full = pd.Series(False, index=s.index)
    full.loc[mask.index] = mask.values
    return full

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

rows = []
for c in num_cols:
    s = df[c]
    im_iqr = iqr_outlier_mask(s)
    im_zsc = zscore_outlier_mask(s)
    rows.append((c, int(im_iqr.sum()), int(im_zsc.sum())))

outlier_table = pd.DataFrame(rows, columns=["feature","iqr_outliers","zscore_outliers"]).sort_values("iqr_outliers", ascending=False)
display(outlier_table)

# --- Handling ---
def winsorize_series(s: pd.Series, lower_q=0.01, upper_q=0.99):
    s_num = pd.to_numeric(s, errors='coerce')
    lo = s_num.quantile(lower_q)
    hi = s_num.quantile(upper_q)
    return s_num.clip(lower=lo, upper=hi)

df_mod = df.copy()

for c in [COLS.get("price"), COLS.get("price_per_sqft"), COLS.get("area"), COLS.get("carpet_area")]:
    if c and c in df_mod.columns:
        df_mod[c] = winsorize_series(df_mod[c])

for c in [COLS.get("bedroom"), COLS.get("floor_num"), COLS.get("super_built_up_area")]:
    if c and c in df_mod.columns:
        s = pd.to_numeric(df_mod[c], errors='coerce')
        q1, q3 = np.percentile(s.dropna(), [25, 75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
        keep = (s.between(lo, hi)) | s.isna()
        df_mod = df_mod.loc[keep].copy()

# Built-up area extreme fix → set absurdly large values to NaN then mean-impute
bua = COLS.get("built_up_area")
if bua and bua in df_mod.columns:
    s = pd.to_numeric(df_mod[bua], errors='coerce')
    # heuristic: top 0.1% considered absurd
    thresh = s.quantile(0.999)
    df_mod.loc[s > thresh, bua] = np.nan
    df_mod[bua] = df_mod[bua].fillna(df_mod[bua].mean())

# Binary features → ensure 0/1
for c in [COLS.get("pooja_room"), COLS.get("study_room"), COLS.get("storeroom"), COLS.get("servant_room")]:
    if c and c in df_mod.columns:
        df_mod[c] = (pd.to_numeric(df_mod[c], errors='coerce').fillna(0) > 0).astype(int)

# Compare outliers again after handling
rows2 = []
num_cols2 = df_mod.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols2:
    s = df_mod[c]
    im_iqr = iqr_outlier_mask(s)
    im_zsc = zscore_outlier_mask(s)
    rows2.append((c, int(im_iqr.sum()), int(im_zsc.sum())))

outlier_table_after = pd.DataFrame(rows2, columns=["feature","iqr_outliers","zscore_outliers"]).sort_values("iqr_outliers", ascending=False)
display(outlier_table_after)

## 24) Save Artifacts

In [ ]:

# Save cleaned CSV (post-outlier-handling version)
out_path = Path("Gurgaon_RealEstate_cleaned.csv")
df_mod.to_csv(out_path, index=False)
print("Saved cleaned CSV to:", out_path.resolve())

# 🔧 Advanced EDA Add‑Ons (v2)
Extra diagnostics and visuals on top of your consolidated analysis:
1. Reproducibility & environment printout
2. Data dictionary (types, missing %, examples)
3. Validation rules & anomaly checks (price/area/ppsf)
4. Rare-category handling for more columns
5. Correlation heatmap (numeric)
6. Market segmentation pivots (median price by Sector × Bedrooms)
7. Robust MAD outliers for key targets (Price, PPSF)
8. Optional figure saving helper
9. Optional quick linear model (if `statsmodels` exists)
10. Optional time fields detection (auto-handled if any)

## A1) Reproducibility — random seed & package versions

In [ ]:

import numpy as np, pandas as pd, matplotlib
np.random.seed(42)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
try:
    import scipy
    print("scipy:", scipy.__version__)
except Exception:
    print("scipy: (not installed)")
try:
    import statsmodels
    print("statsmodels:", statsmodels.__version__)
except Exception:
    print("statsmodels: (not installed)")

## A2) Data Dictionary — types, missing %, example values

In [ ]:

from collections import OrderedDict
def data_dictionary(df: pd.DataFrame, sample_rows=3):
    rows = []
    for c in df.columns:
        s = df[c]
        ex_vals = s.dropna().unique()[:sample_rows]
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "n_unique": int(s.nunique(dropna=True)),
            "missing_count": int(s.isna().sum()),
            "missing_pct": round(float(s.isna().mean()*100), 2),
            "examples": ", ".join(map(lambda x: str(x)[:40], ex_vals))
        })
    dd = pd.DataFrame(rows).sort_values(["missing_pct","column"], ascending=[False, True])
    return dd

DD = data_dictionary(df)  # uses df from earlier sections
display(DD.head(50))
DD.to_csv("data_dictionary.csv", index=False)
print("Saved data_dictionary.csv")

## A3) Data Validation — logical checks & anomalies

In [ ]:

# Auto-detect columns from earlier mapping (COLS). Fall back safely if missing.
price = COLS.get("price")
ppsf  = COLS.get("price_per_sqft")
bua   = COLS.get("built_up_area")
car   = COLS.get("carpet_area")
sba   = COLS.get("super_built_up_area")

def to_num(x): return pd.to_numeric(x, errors="coerce")

issues = []

# Non-negative checks
for c in [price, ppsf, bua, car, sba]:
    if c and c in df.columns:
        neg = (to_num(df[c]) < 0).sum()
        if neg > 0:
            issues.append(f"Column '{c}' has {neg} negative values.")

# Area consistency: sba >= bua >= car (where all present)
if all([sba in df.columns if sba else False, bua in df.columns if bua else False, car in df.columns if car else False]):
    xyz = df[[sba, bua, car]].dropna()
    bad_order = ((to_num(xyz[sba]) < to_num(xyz[bua])) | (to_num(xyz[bua]) < to_num(xyz[car]))).sum()
    if bad_order > 0:
        issues.append(f"Area ordering violated in {bad_order} rows (expected super_built_up ≥ built_up ≥ carpet).")

# Price vs PPSF vs Area sanity: price ≈ ppsf * built_up_area (when both present)
if price and ppsf and bua and all([(c in df.columns) for c in [price, ppsf, bua]]):
    p = to_num(df[price]); q = to_num(df[ppsf]); a = to_num(df[bua])
    m = p.notna() & q.notna() & a.notna() & (a > 0) & (q > 0)
    implied = q[m] * a[m]
    rel_err = (p[m] - implied).abs() / (implied.replace(0, np.nan))
    high_err = rel_err > 0.35  # 35% tolerance (tune)
    n_high = int(high_err.sum())
    if n_high > 0:
        issues.append(f"{n_high} rows have |price - ppsf*built_up_area| > 35% (potential unit/listing anomalies).")
    print("Implied price check — median relative error:", round(rel_err.median(), 3))

if issues:
    print("⚠️ Validation issues found:")
    for x in issues: print(" -", x)
else:
    print("✅ Validation checks passed without flagged issues.")

## A4) Rare-category handling — sector & furnishing (like society)

In [ ]:

def apply_rare_bucketing(df: pd.DataFrame, col: str, threshold: int, other_label: str):
    if col not in df.columns: return df, None
    vc = df[col].astype('category').value_counts()
    rare_mask = df[col].map(vc).fillna(0) <= threshold
    new_col = f"{col}_filtered"
    df[new_col] = df[col].where(~rare_mask, other_label)
    return df, vc

df, sec_vc = apply_rare_bucketing(df, COLS.get("sector") or "sector", threshold=10, other_label="Other Sector")
df, fur_vc = apply_rare_bucketing(df, COLS.get("furnishing_type") or "furnishing_type", threshold=15, other_label="Other Furnishing")

# Plot top 15 for each (if present)
for col in ["sector_filtered", "furnishing_type_filtered"]:
    if col in df.columns:
        vc = df[col].value_counts().head(15)
        plt.figure()
        vc.sort_values(ascending=False).plot(kind="bar")
        plt.title(f"Top-15 — {col}")
        plt.xlabel(col); plt.ylabel("Count")
        plt.tight_layout(); plt.show(); plt.close()

## A5) Correlation heatmap (numeric)

In [ ]:

num_df = df.select_dtypes(include=[np.number])
if num_df.shape[1] >= 2:
    corr = num_df.corr(numeric_only=True)
    plt.figure(figsize=(10,7))
    im = plt.imshow(corr.values, interpolation='nearest')
    plt.title("Correlation Heatmap (numeric)")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show(); plt.close()
else:
    print("Not enough numeric columns for heatmap.")

## A6) Market segmentation — median price by Sector × Bedrooms

In [ ]:

price = COLS.get("price")
sector = COLS.get("sector")
bed = COLS.get("bedroom")
if all([price and price in df.columns, sector and sector in df.columns, bed and bed in df.columns]):
    pv = pd.pivot_table(df, values=price, index=sector, columns=bed, aggfunc='median')
    display(pv.head(20))
    # Top 10 sectors by listing count — median price bars
    top10_sectors = df[sector].value_counts().head(10).index
    med = df[df[sector].isin(top10_sectors)].groupby(sector)[price].median().sort_values(ascending=False)
    plt.figure()
    med.plot(kind="bar")
    plt.title("Median Price by Sector (Top 10 by volume)")
    plt.xlabel("Sector"); plt.ylabel("Median Price")
    plt.tight_layout(); plt.show(); plt.close()
else:
    print("Required columns missing — skipping segmentation.")

## A7) Robust outliers via MAD (Price & PPSF)

In [ ]:

def mad_based_outliers(s: pd.Series, thresh=3.5):
    x = pd.to_numeric(s, errors='coerce').dropna().values
    if x.size < 10:
        return 0
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-9
    z = 0.6745 * (x - med) / mad
    return int((np.abs(z) > thresh).sum())

for key in ["price","price_per_sqft"]:
    col = COLS.get(key)
    if col and col in df.columns:
        n = mad_based_outliers(df[col])
        print(f"{key} ({col}) — MAD outliers >", 3.5, ":", n)

## A8) Optional: Save all open figures to ./figs

In [ ]:

from pathlib import Path
def save_current_fig(name: str, dpi=160):
    Path("figs").mkdir(exist_ok=True)
    import matplotlib.pyplot as plt
    plt.gcf().savefig(Path("figs")/f"{name}.png", dpi=dpi)
    print("Saved:", Path("figs")/f"{name}.png")

## A9) Quick linear model (if statsmodels installed)

In [ ]:

price = COLS.get("price")
bua   = COLS.get("built_up_area")
bed   = COLS.get("bedroom")
bath  = COLS.get("bathroom")

try:
    import statsmodels.api as sm
    if all([price and price in df.columns, bua and bua in df.columns]):
        tmp = pd.DataFrame({
            "price": pd.to_numeric(df[price], errors='coerce'),
            "bua": pd.to_numeric(df[bua], errors='coerce'),
            "bed": pd.to_numeric(df[bed], errors='coerce') if bed and bed in df.columns else np.nan,
            "bath": pd.to_numeric(df[bath], errors='coerce') if bath and bath in df.columns else np.nan,
        }).dropna()
        X = tmp[["bua","bed","bath"]].fillna(0.0)
        X = sm.add_constant(X)
        y = tmp["price"]
        model = sm.OLS(y, X).fit()
        print(model.summary())
    else:
        print("Not enough columns for the quick linear model.")
except Exception as e:
    print("statsmodels not installed or failed to run quick model:", e)

## A10) Optional — Time fields detection and monthly trends (if any)

In [ ]:

date_cols = []
for c in df.columns:
    if "date" in c or "posted" in c or "listed" in c:
        date_cols.append(c)

if date_cols:
    for c in date_cols:
        s = pd.to_datetime(df[c], errors='coerce')
        if s.notna().sum() > 0:
            month = s.dt.to_period('M').dt.to_timestamp()
            cnt = month.value_counts().sort_index()
            plt.figure()
            plt.plot(cnt.index, cnt.values)
            plt.title(f"Listing Counts per Month — {c}")
            plt.xlabel("Month"); plt.ylabel("Count")
            plt.tight_layout(); plt.show(); plt.close()
else:
    print("No date-like columns detected — skipping time trends.")